# ASSIGNMENT 04: TRIỂN KHAI MẠNG CNN TRÊN PYTORCH
**Học phần:** Phát triển các hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

Triển khai mạng tích chập 1D và 2D trên PyTorch cho:
1. Dữ liệu y tế Diabetes (1D Tabular)
2. Dữ liệu định giá nhà House Price (1D Tabular)
3. Dữ liệu đánh giá Customer Reviews (1D Text NLP)
4. Dữ liệu ảnh chữ số viết tay MNIST (2D Image)


In [ ]:
# Cell [1]: Khai báo kiến trúc CNN 1D cho Bài toán Tiểu đường (Slide 10)
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

class DiabetesCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: (batch, 1, 8)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(16 * 4, 8)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.pool(self.relu2(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.relu3(self.fc1(x))
        return self.fc2(x)

model_diab = DiabetesCNN1D()
total_params = sum(p.numel() for p in model_diab.parameters() if p.requires_grad)
print("Kiến trúc mô hình Diabetes CNN 1D:")
print(model_diab)
print(f"Tổng số tham số học được (Trainable Parameters): {total_params:,}")


In [ ]:
# Cell [2]: Kiểm tra kích thước Tensor trung gian (Inspecting Shapes - Tutorial 42)
x_dummy = torch.randn(2, 1, 8)
print("Input shape:       ", x_dummy.shape)
x_c1 = model_diab.relu1(model_diab.conv1(x_dummy))
print("After Conv1+ReLU:  ", x_c1.shape)
x_c2 = model_diab.relu2(model_diab.conv2(x_c1))
print("After Conv2+ReLU:  ", x_c2.shape)
x_p = model_diab.pool(x_c2)
print("After MaxPool1d:   ", x_p.shape)
x_flat = torch.flatten(x_p, 1)
print("After Flatten:     ", x_flat.shape)
out = model_diab(x_dummy)
print("Final Output shape:", out.shape)


In [ ]:
# Cell [3]: Huấn luyện mô hình PyTorch CNN trên dữ liệu Diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv('../data_a4/diabetes.csv')
X = StandardScaler().fit_transform(df.drop('Outcome', axis=1).values)
y = df['Outcome'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr).unsqueeze(1), torch.FloatTensor(y_tr).unsqueeze(1)), batch_size=32, shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_diab.parameters(), lr=0.003)

for epoch in range(1, 16):
    model_diab.train()
    total_loss = 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_diab(bx), by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
    if epoch % 3 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/15] | Loss: {total_loss/len(X_tr):.4f}")
print("Huấn luyện PyTorch CNN hoàn tất!")
